kernel : Python (Pixi)

In [23]:
import anndata
import gc
import numpy as np
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_mtx(mtx_file: str, col_data_file: str, row_data_file: str, series: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    obs_df = pd.read_csv(col_data_file, sep=",", index_col=2)
    var_df = pd.read_csv(row_data_file, sep=",", index_col=0)
    adata.obs = obs_df
    adata.var = var_df

    # Converting to csr_matrix for memory efficiency and speed
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)

    adata.obs["series"] = series
    return adata

#### Main Dataset

In [24]:
MTX_FILE = "ms_lesions_snRNAseq_cleaned_counts_matrix_2023-09-12.mtx.gz"
COL_DATA_FILE = "ms_lesions_snRNAseq_col_data_2023-09-12.txt.gz"
ROW_DATA_FILE = "ms_lesions_snRNAseq_row_data_2023-09-12.txt.gz"
SERIES = "macnair"
SAVE_FILE_NAME = SERIES + ".h5ad"

data_location = os.path.join(raw_count_matrix_location, "macnair")

adata = load_mtx(
    os.path.join(data_location, MTX_FILE),
    os.path.join(data_location, COL_DATA_FILE),
    os.path.join(data_location, ROW_DATA_FILE),
    series = SERIES
)

In [ ]:
adata = adata[adata.obs["exclude_pseudobulk"] == False].copy()
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["celltype"] = adata.obs["type_broad"]
adata.obs["celltype_fine"] = adata.obs["type_fine"]
adata.obs["sample"] = adata.obs["sample_id_anon"]
adata.obs["donor"] = adata.obs["individual_id_anon"]
adata.obs["lesion"] = adata.obs["lesion_type"]
adata.obs["pmi"] = adata.obs["pmi_cat2"]
adata.obs["source"] = adata.obs["sample_source"]
adata.obs["age_scale"] = adata.obs["age_scale"]
adata.obs["sex"] = adata.obs["sex"]
adata.obs["diagnosis"] = adata.obs["diagnosis"]
adata.obs["matter"] = adata.obs["matter"]
adata.obs["pool"] = adata.obs["seq_pool"]
adata.obs["series"] = adata.obs["series"]

keep = ["celltype", "celltype_fine", "sample", "donor", "lesion", "pmi", "source", "age_scale", "sex", "diagnosis", "matter", "pool", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [31]:
ensg_to_symbol = get_ensg_to_symbol()

vn = pd.Index(adata.var_names)
s = vn.to_series(index=vn)
parts = s.str.rsplit("_", n=1, expand=True)

sym_guess = parts[0].astype("string")
ensg_raw  = parts[1].astype("string")

bad = ~ensg_raw.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:")
    print(vn[bad][:20].tolist())

ensg = ensg_raw.str.split(".", n=1).str[0]

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(sym_guess)

adata.var["orig_var_name"] = vn.astype(str).values
adata.var_names = mapped.values.astype(str).tolist()
adata.var_names_make_unique()

In [32]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, "macnair.h5ad"))
del adata
gc.collect()

814

#### Validation Dataset

In [12]:
MTX_FILE = "ms_lesions_snRNAseq_validation_cleaned_counts_matrix_2023-09-12.mtx.gz"
COL_DATA_FILE = "ms_lesions_snRNAseq_validation_col_data_2023-09-12.txt.gz"
ROW_DATA_FILE = "ms_lesions_snRNAseq_validation_row_data_2023-09-12.txt.gz"
SERIES = "macnair_validation"
SAVE_FILE_NAME = SERIES + ".h5ad"

data_location = os.path.join(raw_count_matrix_location, "macnair")

adata = load_mtx(
    os.path.join(data_location, MTX_FILE),
    os.path.join(data_location, COL_DATA_FILE),
    os.path.join(data_location, ROW_DATA_FILE),
    series = SERIES
)

/tmp/ipykernel_356388/1074250388.py:20: DtypeWarning: Columns (0: age_at_death, 1: lesion_type) have mixed types. Specify dtype option on import or set low_memory=False.
  obs_df = pd.read_csv(col_data_file, sep=",", index_col=2)


In [13]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["source"] = adata.obs["study"]
adata.obs["sample"] = adata.obs["sample_id"]
adata.obs["donor"] = adata.obs["donor_id"]
adata.obs["celltype"] = adata.obs["type_broad"]
adata.obs["lesion"] = adata.obs["lesion_type"]
adata.obs["diagnosis"] = adata.obs["diagnosis"]
adata.obs["series"] = adata.obs["series"]
adata.obs["sex"] = adata.obs["sex"]
cat_columns = [
    "source", "sample", "donor", "celltype", 
    "lesion", "diagnosis", "series", "sex"
]
for col in cat_columns:
    adata.obs[col] = adata.obs[col].fillna("Unknown").astype("category")

adata.obs["pmi"] = pd.cut(
    adata.obs["pmi_minutes"], bins=[-np.inf, 12 * 60, np.inf], labels=["up_to_12H", "over_12H"] #type: ignore
) 
adata.obs["pmi"] = adata.obs["pmi"].cat.add_categories("Unknown").fillna("Unknown")

adata.obs["age"] = adata.obs["age_at_death"]
age_mean = adata.obs["age"].mean()
age_sd = adata.obs["age"].std()
adata.obs["age"] = adata.obs["age"].fillna(age_mean)
adata.obs["age_scale"] = (adata.obs["age"] - age_mean) / (2 * age_sd)

keep = ["source", "sample", "donor", "celltype", "pmi", "age_scale", "lesion", "diagnosis", "series", "sex"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [21]:
ensg_to_symbol = get_ensg_to_symbol()

vn = pd.Index(adata.var_names)
s = vn.to_series(index=vn)
parts = s.str.rsplit("_", n=1, expand=True)

sym_guess = parts[0].astype("string")
ensg_raw  = parts[1].astype("string")

bad = ~ensg_raw.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:")
    print(vn[bad][:20].tolist())

ensg = ensg_raw.str.split(".", n=1).str[0]

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(sym_guess)

adata.var["orig_var_name"] = vn.astype(str).values
adata.var_names = mapped.values.astype(str).tolist()
adata.var_names_make_unique()

In [22]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, "macnair_validation.h5ad"))
del adata
gc.collect()

0